# TensorBoard 实践教程：从入门到可视化
本 Notebook 演示如何在 PyTorch 中使用 TensorBoard 实现：
- 训练与验证曲线可视化
- 模型结构展示
- 图像与特征图显示
- 超参数记录
- 混淆矩阵绘制
- 可视化特征图（Feature Maps）
- Embedding 可视化（高维特征降维）

In [1]:
from torch.utils.tensorboard import SummaryWriter
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np
import os

In [2]:
# === 基础配置 ===
log_dir = "./runs/demo_tensorboard"
os.makedirs(log_dir, exist_ok=True)
writer = SummaryWriter(log_dir)

# === 数据加载 ===
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1000, shuffle=False)

Failed to download (trying next):
HTTP Error 404: Not Found



  0%|          | 0/9912422 [00:00<?, ?it/s]

Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



  0%|          | 0/28881 [00:00<?, ?it/s]

Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



  0%|          | 0/1648877 [00:00<?, ?it/s]

Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



  0%|          | 0/4542 [00:00<?, ?it/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



In [3]:
# === 定义一个简单的 MLP 模型 ===
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
    
    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [4]:
# === 模型结构可视化 ===
dummy_input = torch.randn(1, 1, 28, 28)
writer.add_graph(model, dummy_input)

In [5]:
# === 训练循环 ===
epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        
    train_loss = total_loss / len(train_loader)
    train_acc = correct / len(train_loader.dataset)
    
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Accuracy/train", train_acc, epoch)
    
    # === 验证 ===
    model.eval()
    test_loss = 0
    correct = 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            all_preds.extend(pred.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
    
    test_loss /= len(test_loader)
    test_acc = correct / len(test_loader.dataset)
    writer.add_scalar("Loss/test", test_loss, epoch)
    writer.add_scalar("Accuracy/test", test_acc, epoch)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Test Acc={test_acc:.4f}")

Epoch 1: Train Loss=0.4106, Test Acc=0.9345
Epoch 2: Train Loss=0.1904, Test Acc=0.9480
Epoch 3: Train Loss=0.1365, Test Acc=0.9538


In [ ]:
# === 可视化输入图像与混淆矩阵 ===
images, labels = next(iter(train_loader))
img_grid = make_grid(images[:16], nrow=4, normalize=True)
writer.add_image("MNIST Sample Images", img_grid, 0)

cm = confusion_matrix(all_targets, np.array(all_preds).flatten())
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
writer.add_figure("Confusion Matrix", fig)

# === 超参数记录 ===
writer.add_hparams(
    {'lr': 0.001, 'batch_size': 64, 'epochs': epochs},
    {'accuracy': test_acc, 'loss': test_loss}
)

writer.close()
print("TensorBoard logs saved to:", log_dir)
print("Run: tensorboard --logdir=./runs/demo_tensorboard --port=6006")

TensorBoard logs saved to: ./runs/demo_tensorboard
Run: tensorboard --logdir=./runs/demo_tensorboard --port=6006


In [ ]:
# 可视化特征图（Feature Maps）
images = torch.randn(16, 3, 32, 32)
writer.add_images("Input Images", images, 0)
from torchvision.utils import make_grid

feature_maps = torch.randn(16, 1, 28, 28)
grid = make_grid(feature_maps, nrow=4, normalize=True)
writer.add_image("Feature Maps", grid, 0)

In [ ]:
# Embedding 可视化（高维特征降维）
import numpy as np
features = np.random.rand(100, 128)
labels = [f"class_{i%10}" for i in range(100)]

writer.add_embedding(features, metadata=labels, tag="Embedding_demo")